# Camada Gold — Tabelas Analíticas

Cria tabelas analíticas prontas para consumo a partir da camada Silver:
- **Rankings** por período com ROW_NUMBER()
- **Resumos anuais** com agregações
- **Cruzamentos** entre fontes (Reclamações × Ouvidorias, Bancos × Consórcios)
- **Classificação de risco** combinando volume de reclamações + nota da ouvidoria

## Tabelas Gold

| Tabela | Descrição | Origem |
| --- | --- | --- |
| `ranking_reclamacoes` | Ranking de reclamações por banco e período | Silver `reclamacoes_bacen` |
| `resumo_anual_reclamacoes` | Resumo anual por categoria | Silver `reclamacoes_bacen` |
| `ranking_ouvidorias` | Ranking de qualidade das ouvidorias | Silver `ouvidorias_bacen` |
| `resumo_ouvidorias_anual` | Resumo anual das ouvidorias | Silver `ouvidorias_bacen` |
| `ranking_consorcios` | Ranking de reclamações de consórcios | Silver `consorcios_bacen` |
| `resumo_consorcios_anual` | Resumo anual de consórcios | Silver `consorcios_bacen` |
| `reclamacoes_ouvidorias` | Cruzamento Reclamações × Ouvidorias + risco | Gold `ranking_reclamacoes` + Silver `ouvidorias_bacen` |
| `comparativo_anual` | Comparativo Bancos × Consórcios | Gold `ranking_reclamacoes` + Gold `ranking_consorcios` |

> **Pré-requisito**: execute `02_silver_bacen` antes deste notebook.

## Resumo do Notebook

### Resumo

Este notebook implementa a **camada Gold** do projeto BACEN, criando tabelas analíticas prontas para consumo a partir dos dados já tratados na camada Silver. São gerados rankings, resumos anuais, cruzamentos entre fontes e uma classificação de risco, totalizando **8 tabelas Gold** no schema `workspace.bacen_gold`.

### Objetivo

Transformar os dados padronizados da camada Silver em tabelas analíticas de alto valor, com rankings, agregações anuais e cruzamentos entre fontes distintas (Reclamações, Ouvidorias e Consórcios), possibilitando análise comparativa e classificação de risco por instituição.

### O que foi feito

1. **Ranking de Reclamações** (`ranking_reclamacoes`): padroniza categoria em Top/Demais, unifica o volume de reclamações com `COALESCE` (resolve mudança de schema entre 2017–2023 e 2024+), calcula taxa por milhão de clientes e atribui `ranking_periodo` com `ROW_NUMBER()`.
2. **Resumo Anual de Reclamações** (`resumo_anual_reclamacoes`): agrega por ano e categoria (qtd_instituicoes, total_reclamacoes, indice_medio, total_clientes).
3. **Ranking de Ouvidorias** (`ranking_ouvidorias`): preserva notas (prazo, qualidade, final) e métricas de atendimento; atribui dois rankings — `ranking_nota` e `ranking_volume_reclamacoes`.
4. **Resumo Anual de Ouvidorias** (`resumo_ouvidorias_anual`): médias anuais das notas, totais de respostas, reclamações encerradas e contagem de adesão ao Consumidor.gov.
5. **Ranking de Consórcios** (`ranking_consorcios`): calcula taxa de reclamações por milhão de consorciados e atribui `ranking_volume` e `ranking_indice`.
6. **Resumo Anual de Consórcios** (`resumo_consorcios_anual`): agrega por ano (qtd_administradoras, total_reclamacoes, indice_medio, total_procedentes, total_nao_reguladas).
7. **Cruzamento Reclamações × Ouvidorias** (`reclamacoes_ouvidorias`): `LEFT JOIN` da Gold `ranking_reclamacoes` com a Silver `ouvidorias_bacen` (chave: ano + trimestre = período + nome sem "(conglomerado)"), com `classificacao_risco` em 4 níveis (CRÍTICO, ALTO, MÉDIO, BAIXO).
8. **Comparativo Bancos × Consórcios** (`comparativo_anual`): `UNION ALL` de duas CTEs agregadas por ano, unificando métricas para comparação direta entre as duas fontes.

### Fluxo geral

```text
Silver (reclamacoes_bacen, ouvidorias_bacen, consorcios_bacen)
  │
  ├─▶ ranking_reclamacoes          ──┐
  │                                   ├─▶ reclamacoes_ouvidorias (LEFT JOIN + classificacao_risco)
  ├─▶ ouvidorias_bacen ──────────────┘
  │
  ├─▶ ranking_consorcios ────────────┐
  │                                   ├─▶ comparativo_anual (UNION ALL)
  └─▶ ranking_reclamacoes ───────────┘
  │
  └─▶ resumo_anual_reclamacoes, resumo_ouvidorias_anual, resumo_consorcios_anual (agregações anuais)
```

### Verificação

Cada seção possui queries de validação, e há uma verificação final que conta os registros de todas as 8 tabelas Gold:

- **Reclamações**: contagem das duas tabelas; Top 10 instituições do período mais recente (2026 T2); evolução anual por categoria.
- **Ouvidorias**: *(implícita via queries subsequentes)*
- **Combinado**: cobertura total e % de match com ouvidoria; distribuição por `classificacao_risco`; Top 10 instituições CRÍTICAS.
- **Comparativo**: listagem de todas as linhas de `comparativo_anual` para comparação direta Bancos vs Consórcios.
- **Verificação final**: `UNION ALL` com `COUNT(*)` de todas as 8 tabelas Gold, ordenado por nome da tabela.

> **Pré-requisito**: executar `02_silver_bacen` antes deste notebook.

## 1. Reclamações — Ranking e Resumo Anual

Cria `ranking_reclamacoes` (ranking por período com taxa por milhão de clientes) e `resumo_anual_reclamacoes` (agregação anual por categoria Top/Demais).

In [0]:
%sql
-- ============================================================================
-- CAMADA GOLD - TABELA 1: RANKING DE RECLAMAÇÕES
-- ============================================================================
-- Cria ranking_reclamacoes a partir da Silver. Padroniza categoria em
-- Top/Demais, unifica o volume de reclamações (COALESCE resolve mudança
-- de schema entre 2017-2023 e 2024+), calcula taxa por milhão de
-- clientes e atribui ranking_periodo com ROW_NUMBER(). Cria também
-- resumo_anual_reclamacoes com agregações anuais por categoria
-- (qtd_instituicoes, total_reclamacoes, indice_medio, total_clientes).
-- ============================================================================

CREATE OR REPLACE TABLE workspace.bacen_gold.ranking_reclamacoes AS
SELECT
  ano,
  trimestre,
  -- Padronizar categorias em dois grupos: Top e Demais
  CASE
    WHEN categoria LIKE 'Top%' OR categoria LIKE 'Mais de quatro%' THEN 'Top'
    ELSE 'Demais'
  END AS categoria,
  tipo,
  cnpj_if,
  instituicao_financeira,
  indice,
  -- Volume unificado (resolve mudança de schema em 2024+)
  COALESCE(qtd_total_reclamacoes, qtd_total_reclamacoes_respondidas) AS qtd_reclamacoes,
  qtd_reclamacoes_reguladas_procedentes,
  qtd_reclamacoes_reguladas_outras,
  qtd_reclamacoes_nao_reguladas,
  qtd_total_clientes_ccs_scr,
  qtd_clientes_ccs,
  qtd_clientes_scr,
  qtd_clientes_fgc,
  -- Taxa de reclamações por milhão de clientes
  CASE
    WHEN qtd_total_clientes_ccs_scr > 0
    THEN ROUND(
      CAST(COALESCE(qtd_total_reclamacoes, qtd_total_reclamacoes_respondidas) AS DECIMAL(18, 2))
      / qtd_total_clientes_ccs_scr * 1000000, 2)
    ELSE NULL
  END AS taxa_reclamacoes_por_milhao,
  -- Ranking dentro de cada período
  ROW_NUMBER() OVER (
    PARTITION BY ano, trimestre
    ORDER BY COALESCE(qtd_total_reclamacoes, qtd_total_reclamacoes_respondidas) DESC NULLS LAST
  ) AS ranking_periodo,
  tipo_instituicao,
  data_carga
FROM workspace.bacen_silver.reclamacoes_bacen;

-- ============================================================================
-- CAMADA GOLD - TABELA 2: RESUMO ANUAL POR CATEGORIA
-- ============================================================================

CREATE OR REPLACE TABLE workspace.bacen_gold.resumo_anual_reclamacoes AS
SELECT
  ano,
  CASE
    WHEN categoria LIKE 'Top%' OR categoria LIKE 'Mais de quatro%' THEN 'Top'
    ELSE 'Demais'
  END AS categoria,
  COUNT(DISTINCT instituicao_financeira) AS qtd_instituicoes,
  SUM(COALESCE(qtd_total_reclamacoes, qtd_total_reclamacoes_respondidas)) AS total_reclamacoes,
  ROUND(AVG(indice), 2) AS indice_medio,
  SUM(qtd_total_clientes_ccs_scr) AS total_clientes
FROM workspace.bacen_silver.reclamacoes_bacen
GROUP BY
  ano,
  CASE
    WHEN categoria LIKE 'Top%' OR categoria LIKE 'Mais de quatro%' THEN 'Top'
    ELSE 'Demais'
  END
ORDER BY ano, categoria;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================================
-- VERIFICAÇÃO DA CAMADA GOLD — RECLAMAÇÕES
-- ============================================================================
-- Três consultas de validação: (1) contagem de registros das duas
-- tabelas via UNION ALL; (2) Top 10 instituições do período mais recente
-- (2026 T2) com ranking, taxa por milhão e índice; (3) evolução anual
-- por categoria com qtd_instituicoes, total_reclamacoes e indice_medio.
-- ============================================================================

-- Contagem das duas tabelas
SELECT 'ranking_reclamacoes' AS tabela, COUNT(*) AS total_registros
FROM workspace.bacen_gold.ranking_reclamacoes
UNION ALL
SELECT 'resumo_anual_reclamacoes', COUNT(*)
FROM workspace.bacen_gold.resumo_anual_reclamacoes;

-- Top 10 instituições com mais reclamações em 2026 T2
SELECT ano, trimestre, ranking_periodo, categoria, instituicao_financeira,
       qtd_reclamacoes, taxa_reclamacoes_por_milhao, indice
FROM workspace.bacen_gold.ranking_reclamacoes
WHERE ano = 2026 AND trimestre = 2
ORDER BY ranking_periodo
LIMIT 10;

-- Evolução anual por categoria padronizada
SELECT ano, categoria, qtd_instituicoes, total_reclamacoes,
       indice_medio, total_clientes
FROM workspace.bacen_gold.resumo_anual_reclamacoes
ORDER BY ano, categoria;

ano,categoria,qtd_instituicoes,total_reclamacoes,indice_medio,total_clientes
2017,Demais,116,38468,61.04,184088345
2017,Top,12,192434,17.01,1506323401
2018,Demais,120,42942,109.35,185688884
2018,Top,12,187712,14.76,1585268608
2019,Demais,119,45264,182.16,190258162
2019,Top,16,183970,22.53,1736251228
2020,Demais,125,64541,203.69,236560153
2020,Top,19,270853,36.33,1979448806
2021,Demais,156,64780,489.09,448607058
2021,Top,16,203733,37.29,2250145378


## 2. Ouvidorias — Ranking e Resumo Anual

Cria `ranking_ouvidorias` (ranking por nota final e volume de reclamações) e `resumo_ouvidorias_anual` (médias anuais das notas e totais).

In [0]:
%sql
-- ============================================================================
-- CAMADA GOLD OUVIDORIAS - RANKING E RESUMO (CTAS)
-- ============================================================================
-- Cria ranking_ouvidorias a partir da Silver: padroniza categoria em
-- Top/Demais, preserva todas as notas (prazo, qualidade, final) e
-- métricas de atendimento, e atribui dois rankings com ROW_NUMBER():
-- ranking_nota (por nota_final DESC) e ranking_volume_reclamacoes (por
-- qtd_reclamacoes_proc_ouvidoria DESC). Cria também resumo_ouvidorias_anual
-- com médias anuais das notas, totais de respostas, reclamações
-- encerradas e contagem de adesão ao Consumidor.gov.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.ranking_ouvidorias AS
SELECT ano, periodo, tipo_periodo,
  CASE WHEN categoria LIKE 'Top%' THEN 'Top' ELSE 'Demais' END AS categoria,
  tipo, cnpj_if, instituicao_financeira,
  num_respostas_fornecidas, num_respostas_atrasadas, tempo_medio_respostas,
  qtd_reclamacoes_proc_qualid, qtd_reclamacoes_encerradas, qtd_reclamacoes_proc_ouvidoria,
  adesao_consumidor_gov, nota_prazo, nota_qualidade, nota_final,
  ROW_NUMBER() OVER (PARTITION BY ano, periodo, tipo_periodo ORDER BY nota_final DESC NULLS LAST) AS ranking_nota,
  ROW_NUMBER() OVER (PARTITION BY ano, periodo, tipo_periodo ORDER BY qtd_reclamacoes_proc_ouvidoria DESC NULLS LAST) AS ranking_volume_reclamacoes,
  data_carga
FROM workspace.bacen_silver.ouvidorias_bacen;

CREATE OR REPLACE TABLE workspace.bacen_gold.resumo_ouvidorias_anual AS
SELECT ano,
  CASE WHEN categoria LIKE 'Top%' THEN 'Top' ELSE 'Demais' END AS categoria,
  COUNT(DISTINCT instituicao_financeira) AS qtd_instituicoes,
  ROUND(AVG(nota_final), 2) AS nota_final_media,
  ROUND(AVG(nota_prazo), 2) AS nota_prazo_media,
  ROUND(AVG(nota_qualidade), 2) AS nota_qualidade_media,
  ROUND(AVG(tempo_medio_respostas), 2) AS tempo_medio_respostas,
  SUM(num_respostas_fornecidas) AS total_respostas,
  SUM(num_respostas_atrasadas) AS total_respostas_atrasadas,
  SUM(qtd_reclamacoes_encerradas) AS total_reclamacoes_encerradas,
  SUM(CASE WHEN adesao_consumidor_gov = 'SIM' THEN 1 ELSE 0 END) AS qtd_adesao_consumidor_gov
FROM workspace.bacen_silver.ouvidorias_bacen
GROUP BY ano, CASE WHEN categoria LIKE 'Top%' THEN 'Top' ELSE 'Demais' END
ORDER BY ano, categoria

num_affected_rows,num_inserted_rows


## 3. Consórcios — Ranking e Resumo Anual

Cria `ranking_consorcios` (ranking por volume e índice) e `resumo_consorcios_anual` (agregação anual).

In [0]:
%sql
-- ============================================================================
-- CAMADA GOLD CONSÓRCIOS - RANKING E RESUMO (CTAS)
-- ============================================================================
-- Cria ranking_consorcios a partir da Silver: calcula taxa de
-- reclamações por milhão de consorciados e atribui dois rankings com
-- ROW_NUMBER(): ranking_volume (por qtd_total_reclamacoes DESC) e
-- ranking_indice (por indice DESC). Cria também resumo_consorcios_anual
-- com agregações anuais (qtd_administradoras, total_reclamacoes,
-- indice_medio, total_procedentes, total_nao_reguladas).
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.ranking_consorcios AS
SELECT ano, semestre, cnpj_ac, administradora, indice,
  qtd_reclamacoes_procedentes, qtd_reclamacoes_outras, qtd_reclamacoes_nao_reguladas,
  qtd_total_reclamacoes, qtd_clientes_consorciados,
  CASE WHEN qtd_clientes_consorciados > 0
    THEN ROUND(CAST(qtd_total_reclamacoes AS DECIMAL(18, 2)) / qtd_clientes_consorciados * 1000000, 2)
    ELSE NULL END AS taxa_reclamacoes_por_milhao,
  ROW_NUMBER() OVER (PARTITION BY ano, semestre ORDER BY qtd_total_reclamacoes DESC NULLS LAST) AS ranking_volume,
  ROW_NUMBER() OVER (PARTITION BY ano, semestre ORDER BY indice DESC NULLS LAST) AS ranking_indice,
  data_carga
FROM workspace.bacen_silver.consorcios_bacen;

CREATE OR REPLACE TABLE workspace.bacen_gold.resumo_consorcios_anual AS
SELECT ano,
  COUNT(DISTINCT administradora) AS qtd_administradoras,
  SUM(qtd_total_reclamacoes) AS total_reclamacoes,
  ROUND(AVG(indice), 2) AS indice_medio,
  SUM(qtd_clientes_consorciados) AS total_clientes,
  SUM(qtd_reclamacoes_procedentes) AS total_procedentes,
  SUM(qtd_reclamacoes_nao_reguladas) AS total_nao_reguladas
FROM workspace.bacen_silver.consorcios_bacen
GROUP BY ano ORDER BY ano

num_affected_rows,num_inserted_rows


## 4. Combinado: Reclamações × Ouvidorias

Cruza os dados de **Reclamações** (reclamações de clientes) com **Ouvidorias** (qualidade do atendimento) por instituição e período.

**Join key**: `ano + trimestre = periodo + REPLACE(instituicao_financeira, ' (conglomerado)', '')`
**Cobertura**: 37,3% dos registros de Reclamações têm match com Ouvidorias (2017-2021)
**Classificação de risco**: CRÍTICO, ALTO, MÉDIO, BAIXO — combina volume de reclamações + nota da ouvidoria

In [0]:
%sql
-- ============================================================================
-- GOLD COMBINADO: RECLAMAÇÕES × OUVIDORIAS
-- ============================================================================
-- Cria reclamacoes_ouvidorias com LEFT JOIN da Gold ranking_reclamacoes
-- com a Silver ouvidorias_bacen. Mantém todas as Reclamações (2017-2026)
-- e traz Ouvidorias quando há match (2017-2021). Join key: ano +
-- trimestre=periodo + nome sem "(conglomerado)". Calcula classificacao_risco
-- combinando volume de reclamações + nota da ouvidoria em 4 níveis:
-- CRÍTICO, ALTO, MÉDIO, BAIXO. Registros sem match (2022+) ficam NULL.
-- ============================================================================

-- LEFT JOIN: mantém todas as Reclamações (2017-2026), traz Ouvidorias quando há match (2017-2021)
-- Join key: ano + trimestre=periodo + nome sem "(conglomerado)"

CREATE OR REPLACE TABLE workspace.bacen_gold.reclamacoes_ouvidorias AS
SELECT
  r.ano, r.trimestre, r.categoria, r.tipo, r.cnpj_if, r.instituicao_financeira,
  -- Reclamações
  r.qtd_reclamacoes, r.indice, r.taxa_reclamacoes_por_milhao,
  r.ranking_periodo AS ranking_reclamacoes,
  -- Ouvidorias (NULL quando não há match = 2022+)
  o.nota_final AS nota_ouvidoria, o.nota_prazo AS nota_prazo_ouvidoria,
  o.nota_qualidade AS nota_qualidade_ouvidoria, o.tempo_medio_respostas,
  o.num_respostas_fornecidas, o.num_respostas_atrasadas,
  o.qtd_reclamacoes_encerradas AS qtd_reclamacoes_encerradas_ouvidoria,
  o.adesao_consumidor_gov,
  CASE WHEN o.nota_final IS NOT NULL THEN 1 ELSE 0 END AS tem_dado_ouvidoria,
  -- Classificação de risco combinada
  CASE
    WHEN o.nota_final IS NOT NULL AND r.qtd_reclamacoes IS NOT NULL THEN
      CASE
        WHEN r.qtd_reclamacoes > 1000 AND o.nota_final < 2.0 THEN 'CRÍTICO'
        WHEN r.qtd_reclamacoes > 500  AND o.nota_final < 3.0 THEN 'ALTO'
        WHEN r.qtd_reclamacoes > 100  AND o.nota_final < 3.5 THEN 'MÉDIO'
        ELSE 'BAIXO'
      END
    ELSE NULL
  END AS classificacao_risco,
  r.data_carga
FROM workspace.bacen_gold.ranking_reclamacoes r
LEFT JOIN workspace.bacen_silver.ouvidorias_bacen o
  ON  r.ano = o.ano
  AND r.trimestre = o.periodo
  AND o.tipo_periodo = 'T'
  AND REPLACE(r.instituicao_financeira, ' (conglomerado)', '') = o.instituicao_financeira

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================================
-- VERIFICAÇÃO DO GOLD COMBINADO
-- ============================================================================
-- Três consultas de validação: (1) cobertura total e % de registros
-- com dado de ouvidoria; (2) distribuição por classificacao_risco com
-- média de reclamações e nota; (3) Top 10 instituições CRÍTICAS com
-- qtd_reclamacoes, nota_ouvidoria, notas de prazo/qualidade e taxa por
-- milhão.
-- ============================================================================
-- Cobertura e contagem
SELECT COUNT(*) AS total, SUM(tem_dado_ouvidoria) AS com_ouvidoria,
       ROUND(SUM(tem_dado_ouvidoria) * 100.0 / COUNT(*), 1) AS pct_cobertura
FROM workspace.bacen_gold.reclamacoes_ouvidorias;

-- Distribuição por classificação de risco
SELECT classificacao_risco, COUNT(*) AS qtd,
  ROUND(AVG(qtd_reclamacoes), 0) AS media_reclamacoes,
  ROUND(AVG(nota_ouvidoria), 2) AS media_nota_ouvidoria
FROM workspace.bacen_gold.reclamacoes_ouvidorias
WHERE classificacao_risco IS NOT NULL
GROUP BY classificacao_risco
ORDER BY CASE classificacao_risco WHEN 'CRÍTICO' THEN 1 WHEN 'ALTO' THEN 2 WHEN 'MÉDIO' THEN 3 WHEN 'BAIXO' THEN 4 END;

-- Top 10 instituições CRÍTICAS
SELECT ano, trimestre, instituicao_financeira, qtd_reclamacoes, nota_ouvidoria,
  nota_prazo_ouvidoria, nota_qualidade_ouvidoria, taxa_reclamacoes_por_milhao
FROM workspace.bacen_gold.reclamacoes_ouvidorias
WHERE classificacao_risco = 'CRÍTICO'
ORDER BY qtd_reclamacoes DESC LIMIT 10

ano,trimestre,instituicao_financeira,qtd_reclamacoes,nota_ouvidoria,nota_prazo_ouvidoria,nota_qualidade_ouvidoria,taxa_reclamacoes_por_milhao
2020,4,PAN (conglomerado),7614,1.268199,1.000000,3.336398,1075.13
2019,4,SAFRA (conglomerado),4092,1.300602,1.000000,2.201204,1638.00
2021,1,PAN (conglomerado),3953,1.351049,1.060797,2.841302,443.76
2021,2,BTG PACTUAL/BANCO PAN (conglomerado),3630,1.483294,1.439867,3.126721,288.63
2021,4,C6 BANK (conglomerado),3311,1.985367,1.516316,2.654417,243.56
2019,2,INTER (conglomerado),2205,1.581330,1.000000,2.762660,874.71
2019,3,INTER (conglomerado),1753,1.853019,1.103300,3.402738,552.73
2020,4,INTER (conglomerado),1634,1.000000,1.019697,3.694410,202.48
2020,3,BNP PARIBAS (conglomerado),1392,1.397295,2.594590,1.000000,534.90
2020,4,BNP PARIBAS (conglomerado),1136,1.959376,1.862046,2.656707,449.95


## 5. Comparativo: Bancos × Consórcios

Compara reclamações entre **Bancos/Financeiras** (trimestral) e **Consórcios** (semestral) ao longo dos anos.

**Observação**: Não há join por entidade — CNPJ e nomes são distintos entre as duas fontes. O cruzamento é feito no nível anual, unificando métricas para comparação.
**Cobertura**: Bancos 2017-2026 | Consórcios 2014-2026

In [0]:
%sql
-- ============================================================================
-- GOLD COMPARATIVO: RECLAMAÇÕES BANCOS × CONSÓRCIOS
-- ============================================================================
-- Cria comparativo_anual com UNION ALL de duas CTEs: rec_anual agrega
-- ranking_reclamacoes por ano (fonte Bancos, periodicidade Trimestral) e
-- con_anual agrega ranking_consorcios por ano (fonte Consórcios,
-- periodicidade Semestral). Cada CTE calcula qtd_instituicoes,
-- total_reclamacoes, indice_medio, total_clientes e taxa_por_milhao.
-- Resultado: tabela única para comparação ano a ano entre os dois tipos.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.comparativo_anual AS
WITH rec_anual AS (
  SELECT 'Bancos e Financeiras' AS fonte, ano, 'Trimestral' AS periodicidade,
    COUNT(DISTINCT instituicao_financeira) AS qtd_instituicoes,
    SUM(qtd_reclamacoes) AS total_reclamacoes,
    ROUND(AVG(indice), 2) AS indice_medio,
    SUM(qtd_total_clientes_ccs_scr) AS total_clientes,
    CASE WHEN SUM(qtd_total_clientes_ccs_scr) > 0
      THEN ROUND(SUM(qtd_reclamacoes) * 1000000.0 / SUM(qtd_total_clientes_ccs_scr), 2)
      ELSE NULL END AS taxa_por_milhao
  FROM workspace.bacen_gold.ranking_reclamacoes
  GROUP BY ano
),
con_anual AS (
  SELECT 'Consórcios' AS fonte, ano, 'Semestral' AS periodicidade,
    COUNT(DISTINCT administradora) AS qtd_instituicoes,
    SUM(qtd_total_reclamacoes) AS total_reclamacoes,
    ROUND(AVG(indice), 2) AS indice_medio,
    SUM(qtd_clientes_consorciados) AS total_clientes,
    CASE WHEN SUM(qtd_clientes_consorciados) > 0
      THEN ROUND(SUM(qtd_total_reclamacoes) * 1000000.0 / SUM(qtd_clientes_consorciados), 2)
      ELSE NULL END AS taxa_por_milhao
  FROM workspace.bacen_gold.ranking_consorcios
  GROUP BY ano
)
SELECT * FROM rec_anual
UNION ALL SELECT * FROM con_anual
ORDER BY ano, fonte

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- ============================================================================
-- VERIFICAÇÃO DO GOLD COMPARATIVO
-- ============================================================================
-- Lista todas as linhas de comparativo_anual ordenadas por ano e fonte,
-- mostrando periodicidade, qtd_instituicoes, total_reclamacoes,
-- indice_medio e taxa_por_milhao para comparação direta Bancos vs Consórcios.
-- ============================================================================
SELECT ano, fonte, periodicidade, qtd_instituicoes, total_reclamacoes, indice_medio, taxa_por_milhao
FROM workspace.bacen_gold.comparativo_anual
ORDER BY ano, fonte

ano,fonte,periodicidade,qtd_instituicoes,total_reclamacoes,indice_medio,taxa_por_milhao
2014,Consórcios,Semestral,93,3255,230.68,480.66
2015,Consórcios,Semestral,104,6947,78.06,503.35
2016,Consórcios,Semestral,166,6060,50.48,446.79
2017,Bancos e Financeiras,Trimestral,127,230902,44.33,136.60
2017,Consórcios,Semestral,109,6680,108.03,379.48
2018,Bancos e Financeiras,Trimestral,129,230654,70.24,130.24
2018,Consórcios,Semestral,109,5985,133.74,432.16
2019,Bancos e Financeiras,Trimestral,130,229234,120.33,118.99
2019,Consórcios,Semestral,86,5520,228.02,385.87
2020,Bancos e Financeiras,Trimestral,134,335394,135.42,151.35


In [0]:
%sql
-- ============================================================================
-- VERIFICAÇÃO FINAL — TODAS AS TABELAS GOLD
-- ============================================================================
-- Conta os registros de cada uma das 8 tabelas Gold criadas neste
-- notebook (ranking_reclamacoes, resumo_anual_reclamacoes,
-- ranking_ouvidorias, resumo_ouvidorias_anual, ranking_consorcios,
-- resumo_consorcios_anual, reclamacoes_ouvidorias, comparativo_anual)
-- em uma única consulta UNION ALL ordenada por nome da tabela.
-- ============================================================================
SELECT 'ranking_reclamacoes' AS tabela, COUNT(*) AS registros FROM workspace.bacen_gold.ranking_reclamacoes
UNION ALL SELECT 'resumo_anual_reclamacoes', COUNT(*) FROM workspace.bacen_gold.resumo_anual_reclamacoes
UNION ALL SELECT 'ranking_ouvidorias', COUNT(*) FROM workspace.bacen_gold.ranking_ouvidorias
UNION ALL SELECT 'resumo_ouvidorias_anual', COUNT(*) FROM workspace.bacen_gold.resumo_ouvidorias_anual
UNION ALL SELECT 'ranking_consorcios', COUNT(*) FROM workspace.bacen_gold.ranking_consorcios
UNION ALL SELECT 'resumo_consorcios_anual', COUNT(*) FROM workspace.bacen_gold.resumo_consorcios_anual
UNION ALL SELECT 'reclamacoes_ouvidorias', COUNT(*) FROM workspace.bacen_gold.reclamacoes_ouvidorias
UNION ALL SELECT 'comparativo_anual', COUNT(*) FROM workspace.bacen_gold.comparativo_anual
ORDER BY tabela

tabela,registros
comparativo_anual,23
ranking_consorcios,2015
ranking_ouvidorias,3233
ranking_reclamacoes,5060
reclamacoes_ouvidorias,5060
resumo_anual_reclamacoes,20
resumo_consorcios_anual,13
resumo_ouvidorias_anual,7
